-*- coding: utf-8 -*-

## Check GPU

In [ ]:
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ No GPU - will use CPU (slower)")

## Import Libraries

In [ ]:
import numpy as np
import supervision as sv
from ultralytics import YOLO
import time
import os
import subprocess
from IPython.display import Video, display

print("✅ All imports successful")

## Mount Drive & Load Model

In [ ]:
drive.mount('/content/drive')

DRIVE_FOLDER = '/content/drive/MyDrive/bee-monitoring'
model_path = f'{DRIVE_FOLDER}/yolo11m_bee_best.onnx'

# Load model
if os.path.exists(model_path):
    model = YOLO(model_path, task='detect')
    print(f"✅ YOLO11m model loaded on {'cuda:0' if torch.cuda.is_available() else 'cpu'}")
else:
    print(f"❌ Model not found at: {model_path}")

# Load logo
logo_path = f'{DRIVE_FOLDER}/innovation-lab-logo.png'
if os.path.exists(logo_path):
    logo = cv2.imread(logo_path, cv2.IMREAD_UNCHANGED)
    logo_height, logo_width = logo.shape[:2]
    new_width = 275
    new_height = int(logo_height * (new_width / logo_width))
    logo_resized = cv2.resize(logo, (new_width, new_height), interpolation=cv2.INTER_LANCZOS4)
    
    # Create shadow
    logo_shadow = np.zeros_like(logo_resized)
    if logo_resized.shape[2] == 4:
        logo_shadow[:, :, :3] = 30
        logo_shadow[:, :, 3] = logo_resized[:, :, 3] * 0.5
    
    print(f"✅ Logo loaded: {new_width}x{new_height}")
else:
    logo_resized = None
    logo_shadow = None

## Select & Convert Video (FAST!)

In [ ]:
input_video = f'{DRIVE_FOLDER}/clean_bee_hi_res.mp4'

# Fast conversion: 4K → 1080p (30 seconds instead of 15 minutes!)
print("🔄 Converting to 1080p (optimized)...")
compatible_video = 'bee_1080p.mp4'

cmd = [
    'ffmpeg', '-i', input_video,
    '-vf', 'scale=1920:1080',
    '-c:v', 'h264_nvenc',  # GPU encoder
    '-preset', 'p4',
    '-crf', '23',
    '-c:a', 'copy',
    '-y',
    compatible_video
]

result = subprocess.run(cmd, capture_output=True, text=True)

if result.returncode == 0:
    cap = cv2.VideoCapture(compatible_video)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()
    
    print(f"✅ Video ready!")
    print(f"   Resolution: {width}x{height}")
    print(f"   FPS: {fps}")
    print(f"   Frames: {total_frames}")
    
    input_video = compatible_video
else:
    print("❌ Conversion failed, using original")

## Configuration

In [ ]:
    'output_path': 'bee_enhanced.mp4',
    'conf_threshold': 0.50,
    'iou_threshold': 0.45,
    'track_activation_threshold': 0.25,
    'lost_track_buffer': 30,
    'minimum_matching_threshold': 0.8,
    'minimum_consecutive_frames': 1,
    'show_trails': True,
    'trail_length': 30,
    'show_labels': True,
    'use_bytetrack': True,
}

print("✅ Configuration ready")

## FPS-Aware Behavior Classifier (FIXED!)

In [ ]:
    """
    FPS-aware behavior classification - works at ANY frame rate!
    """
    
    if len(track_positions) < int(0.5 * fps):
        return 'unknown'
    
    positions = np.array([(x, y) for x, y, _ in track_positions])
    
    # 1. Speed (pixels per frame)
    distances = np.sqrt(np.sum(np.diff(positions, axis=0)**2, axis=1))
    avg_speed = np.mean(distances)
    
    # 2. Direction changes
    if len(positions) > 2:
        vectors = np.diff(positions, axis=0)
        angles = np.arctan2(vectors[:, 1], vectors[:, 0])
        angle_changes = np.abs(np.diff(angles))
        angle_changes = np.minimum(angle_changes, 2*np.pi - angle_changes)
        avg_angle_change = np.mean(angle_changes)
    else:
        avg_angle_change = 0
    
    # 3. Straightness
    total_path = np.sum(distances)
    displacement = np.linalg.norm(positions[-1] - positions[0])
    straightness = displacement / total_path if total_path > 0 else 0
    
    # FPS-AWARE THRESHOLDS (automatically scale!)
    fps_scale = 30.0 / fps
    SPEED_THRESHOLD_FAST = 3.0 * fps_scale
    SPEED_THRESHOLD_SLOW = 1.0 * fps_scale
    ANGLE_THRESHOLD_ERRATIC = 0.5
    STRAIGHTNESS_THRESHOLD = 0.6
    
    # Classify
    if avg_speed < SPEED_THRESHOLD_SLOW:
        return 'browsing'
    elif avg_speed > SPEED_THRESHOLD_FAST:
        if avg_angle_change > ANGLE_THRESHOLD_ERRATIC or straightness < STRAIGHTNESS_THRESHOLD:
            return 'erratic'
        else:
            return 'flying'
    else:
        if avg_angle_change > ANGLE_THRESHOLD_ERRATIC:
            return 'erratic'
        else:
            return 'browsing'

print("✅ FPS-aware behavior classifier ready")

## Enhanced Processing Function

In [ ]:
    """
    Enhanced processing with:
    - FPS-aware behavior classification
    - Logo with fade-in + shadow
    - 5-second rolling average
    - Text symbols (no emoji issues)
    """
    
    cap = cv2.VideoCapture(input_path)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))
    
    # Initialize ByteTrack
    byte_tracker = sv.ByteTrack(
        track_activation_threshold=config['track_activation_threshold'],
        lost_track_buffer=config['lost_track_buffer'],
        minimum_matching_threshold=config['minimum_matching_threshold'],
        minimum_consecutive_frames=config['minimum_consecutive_frames'],
        frame_rate=int(fps)
    )
    
    # Annotators
    box_annotator = sv.BoxAnnotator(thickness=2)
    label_annotator = sv.LabelAnnotator(text_scale=0.5, text_thickness=2)
    trace_annotator = sv.TraceAnnotator(thickness=2, trace_length=config['trail_length'])
    
    # Logo setup
    logo_position = None
    shadow_position = None
    if logo is not None:
        logo_h, logo_w = logo.shape[:2]
        padding = 20
        shadow_offset = 5
        logo_position = (width - logo_w - padding, height - logo_h - padding)
        shadow_position = (logo_position[0] + shadow_offset, logo_position[1] + shadow_offset)
    
    fade_duration_frames = int(2 * fps)
    
    # Tracking
    track_history = {}
    track_behaviors = {}
    
    # Rolling average
    rolling_window_frames = int(5 * fps)
    bee_count_history = []
    
    print(f"\n🎬 Processing with FPS-aware behavior analysis...")
    print("="*70)
    
    frame_count = 0
    start_time = time.time()
    inference_times = []
    behavior_stats = {'flying': 0, 'erratic': 0, 'browsing': 0}
    
    try:
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            
            frame_count += 1
            
            # YOLO inference
            t0 = time.time()
            results = model(frame, conf=config['conf_threshold'], iou=config['iou_threshold'], verbose=False)[0]
            inference_times.append(time.time() - t0)
            
            detections = sv.Detections.from_ultralytics(results)
            detections = byte_tracker.update_with_detections(detections)
            
            # Update track history and classify
            for bbox, tracker_id in zip(detections.xyxy, detections.tracker_id):
                x1, y1, x2, y2 = bbox
                center_x = (x1 + x2) / 2
                center_y = (y1 + y2) / 2
                
                if tracker_id not in track_history:
                    track_history[tracker_id] = []
                    track_behaviors[tracker_id] = 'unknown'
                
                track_history[tracker_id].append((center_x, center_y, frame_count))
                
                # Keep last 3 seconds
                max_history = int(3 * fps)
                if len(track_history[tracker_id]) > max_history:
                    track_history[tracker_id] = track_history[tracker_id][-max_history:]
                
                # Classify (need 1 second of data)
                if len(track_history[tracker_id]) >= int(1 * fps):
                    track_behaviors[tracker_id] = classify_bee_behavior(track_history[tracker_id], fps)
            
            # Rolling average
            bee_count_history.append(len(detections))
            if len(bee_count_history) > rolling_window_frames:
                bee_count_history = bee_count_history[-rolling_window_frames:]
            rolling_avg = np.mean(bee_count_history) if bee_count_history else 0
            
            # Annotate trails
            if len(detections) > 0:
                frame = trace_annotator.annotate(scene=frame, detections=detections)
            
            # Bounding boxes with behavior colors
            for bbox, tracker_id in zip(detections.xyxy, detections.tracker_id):
                behavior = track_behaviors.get(tracker_id, 'unknown')
                color_map = {
                    'flying': (0, 255, 255),
                    'erratic': (0, 165, 255),
                    'browsing': (0, 255, 0),
                    'unknown': (128, 128, 128)
                }
                color = color_map.get(behavior, (255, 255, 255))
                x1, y1, x2, y2 = bbox.astype(int)
                cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
            
            # Labels with TEXT symbols (no emojis!)
            if len(detections) > 0:
                labels = []
                for tracker_id, confidence in zip(detections.tracker_id, detections.confidence):
                    behavior = track_behaviors.get(tracker_id, 'unknown')
                    symbol_map = {
                        'flying': 'FLY',
                        'erratic': 'ERR',
                        'browsing': 'BRW',
                        'unknown': '???'
                    }
                    symbol = symbol_map.get(behavior, '???')
                    labels.append(f"#{tracker_id} {symbol} {confidence:0.2f}")
                
                frame = label_annotator.annotate(scene=frame, detections=detections, labels=labels)
            
            # Count behaviors
            behavior_stats = {'flying': 0, 'erratic': 0, 'browsing': 0}
            for tracker_id in detections.tracker_id:
                behavior = track_behaviors.get(tracker_id, 'unknown')
                if behavior in behavior_stats:
                    behavior_stats[behavior] += 1
            
            # Stats panel
            panel_height = 200
            cv2.rectangle(frame, (10, 10), (380, panel_height), (0, 0, 0), -1)
            cv2.rectangle(frame, (10, 10), (380, panel_height), (0, 255, 255), 2)
            
            y_pos = 30
            cv2.putText(frame, "DIGITAL4.AI BYTETRACK", (20, y_pos), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)
            y_pos += 30
            cv2.putText(frame, f"BEES NOW: {len(detections)}", (20, y_pos), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
            y_pos += 30
            cv2.putText(frame, f"5s AVG: {rolling_avg:.1f}", (20, y_pos), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (100, 255, 100), 2)
            
            if sum(behavior_stats.values()) > 0:
                y_pos += 25
                cv2.putText(frame, "BEHAVIORS:", (20, y_pos), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (200, 200, 200), 1)
                y_pos += 20
                cv2.putText(frame, f"  Flying: {behavior_stats['flying']}", (20, y_pos), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 255), 1)
                y_pos += 18
                cv2.putText(frame, f"  Erratic: {behavior_stats['erratic']}", (20, y_pos), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 165, 255), 1)
                y_pos += 18
                cv2.putText(frame, f"  Browsing: {behavior_stats['browsing']}", (20, y_pos), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)
            
            y_pos += 25
            cv2.putText(frame, f"FRAME: {frame_count}/{total_frames}", (20, y_pos), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (150, 150, 150), 1)
            
            # Logo with fade-in
            if logo is not None and logo_position is not None:
                fade_alpha = min(1.0, frame_count / fade_duration_frames)
                x, y = logo_position
                shadow_x, shadow_y = shadow_position
                logo_h, logo_w = logo.shape[:2]
                
                # Shadow
                if logo_shadow is not None and logo_shadow.shape[2] == 4:
                    shadow_alpha_channel = (logo_shadow[:, :, 3] / 255.0) * fade_alpha * 0.6
                    for c in range(3):
                        frame[shadow_y:shadow_y+logo_h, shadow_x:shadow_x+logo_w, c] = (
                            shadow_alpha_channel * logo_shadow[:, :, c] +
                            (1 - shadow_alpha_channel) * frame[shadow_y:shadow_y+logo_h, shadow_x:shadow_x+logo_w, c]
                        )
                
                # Logo
                if logo.shape[2] == 4:
                    alpha = (logo[:, :, 3] / 255.0) * fade_alpha
                    for c in range(3):
                        frame[y:y+logo_h, x:x+logo_w, c] = (
                            alpha * logo[:, :, c] +
                            (1 - alpha) * frame[y:y+logo_h, x:x+logo_w, c]
                        )
            
            out.write(frame)
            
            # Progress
            if frame_count % 30 == 0 or frame_count == total_frames:
                elapsed = time.time() - start_time
                fps_processing = frame_count / elapsed
                avg_inference = np.mean(inference_times[-30:]) * 1000
                eta = (total_frames - frame_count) / fps_processing if fps_processing > 0 else 0
                
                print(f"Frame {frame_count:4d}/{total_frames} ({frame_count/total_frames*100:5.1f}%) | "
                      f"Bees:{len(detections):3d} Avg:{rolling_avg:.1f} | "
                      f"FLY:{behavior_stats['flying']} ERR:{behavior_stats['erratic']} BRW:{behavior_stats['browsing']} | "
                      f"GPU:{avg_inference:5.1f}ms | FPS:{fps_processing:5.1f} | ETA:{eta/60:4.1f}min")
    
    except KeyboardInterrupt:
        print("\n⚠️ Processing interrupted")
    
    finally:
        cap.release()
        out.release()
    
    elapsed = time.time() - start_time
    avg_inference = np.mean(inference_times) * 1000 if inference_times else 0
    
    print("\n" + "="*70)
    print(f"✅ PROCESSING COMPLETE!")
    print("="*70)
    print(f"Processed: {frame_count}/{total_frames} frames")
    print(f"⚡ Avg inference: {avg_inference:.1f}ms")
    print(f"⏱️  Total time: {elapsed/60:.1f} minutes")
    print(f"🚀 Processing FPS: {frame_count/elapsed:.1f}")
    print(f"📁 Output: {output_path}")
    
    return output_path, track_history, track_behaviors

print("✅ Processing function ready")

## Run Processing

In [ ]:
    model=model,
    input_path=input_video,
    output_path=CONFIG['output_path'],
    config=CONFIG,
    logo=logo_resized,
    logo_shadow=logo_shadow
)

print(f"\n✅ Enhanced video ready: {output_video}")
print("\n📊 Features included:")
print("  ✅ FPS-aware behavior classification (works at any frame rate!)")
print("  ✅ Logo with 2-second fade-in + shadow")
print("  ✅ 5-second rolling average bee count")
print("  ✅ Text symbols (FLY/ERR/BRW) - no emoji rendering issues")
print("  ✅ Color-coded bounding boxes by behavior")
print("  ✅ Real-time behavior statistics")

## Export Data

In [ ]:

# Get video info
cap = cv2.VideoCapture(input_video)
fps = cap.get(cv2.CAP_PROP_FPS)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
cap.release()

# Export
export_data = {
    'video': os.path.basename(input_video),
    'fps': fps,
    'total_frames': total_frames,
    'tracks': {},
    'summary': {
        'total_unique_bees': len(track_history),
        'behaviors': {
            'flying': sum(1 for b in track_behaviors.values() if b == 'flying'),
            'erratic': sum(1 for b in track_behaviors.values() if b == 'erratic'),
            'browsing': sum(1 for b in track_behaviors.values() if b == 'browsing'),
            'unknown': sum(1 for b in track_behaviors.values() if b == 'unknown')
        }
    }
}

for track_id, positions in track_history.items():
    export_data['tracks'][int(track_id)] = {
        'behavior': track_behaviors.get(track_id, 'unknown'),
        'trajectory': [(float(x), float(y), int(f)) for x, y, f in positions]
    }

json_path = 'bee_tracking_data.json'
with open(json_path, 'w') as f:
    json.dump(export_data, f, indent=2)

print(f"\n📊 Exported tracking data:")
print(f"   Unique bees: {export_data['summary']['total_unique_bees']}")
print(f"   Flying: {export_data['summary']['behaviors']['flying']}")
print(f"   Erratic: {export_data['summary']['behaviors']['erratic']}")
print(f"   Browsing: {export_data['summary']['behaviors']['browsing']}")
print(f"   Unknown: {export_data['summary']['behaviors']['unknown']}")

# Download
from google.colab import files
files.download(json_path)

## Display & Download

In [ ]:
display(Video(output_video, width=800))

print("\n📥 Downloading video...")
files.download(output_video)

print("\n✅ All done!")